# Fire & Smoke Detection — YOLO11 di Google Colab

Notebook end-to-end: Google Drive → validasi/cleaning → split train/val/test → visualisasi anotasi → transfer learning YOLO11n → cross-validation K-Fold tergrup → evaluasi per kelas pada test → ekspor `best.pt`.

## Persiapan dataset (wajib sebelum training)

1. Kumpulkan foto api, asap, keduanya, dan **background tanpa bahaya** dari sudut kamera, jarak, pencahayaan, cuaca, serta siang/malam yang beragam. Hindari frame identik dari video yang sama.
2. Data mentah **tanpa bounding-box berlabel tidak cukup** untuk supervised object detection. Anotasi setiap api/asap dengan Label Studio, CVAT, atau Roboflow; ekspor **YOLO Detection**. Background negatif harus memiliki `.txt` kosong. Format tiap baris: `class_id x_center y_center width height` (koordinat 0–1).
3. Untuk CCTV/video, bagi berdasarkan **video, sesi, atau kamera** sebelum ekspor: rekaman berbeda untuk train, validation, dan test. Jangan membagi frame acak lintas split (data leakage). Notebook mendukung split YOLO yang sudah ada, atau satu folder `images/` + `labels/` berpasangan.
4. Letakkan dataset **yang sudah diekstrak** di `My Drive/visikom/`; Colab memasangnya sebagai `/content/drive/MyDrive/visikom`. Buka **Runtime → Change runtime type → T4 GPU**.

**Algoritma:** YOLO11n detection, supervised learning + transfer learning dari `yolo11n.pt`. Bounding box dilatih langsung, ukuran nano dipilih untuk kecepatan webcam/CCTV. `0=smoke`, `1=fire` kecuali metadata dataset Anda menyatakan ID lain.

## Peta langkah notebook vs materi kuliah

| Tahap notebook | Materi | Isi yang dipakai |
| --- | --- | --- |
| 0-2: setup, split, cleaning, validasi anotasi | Pertemuan 3 - Preprocessing Data | Data cleansing (gambar rusak/noisy/duplikat dibuang), validasi fitur (bbox ternormalisasi 5 kolom), data split train/val/test; mencegah GIGO masuk training |
| 3: training YOLO11n | Pertemuan 4 - Supervised Learning | Belajar dari data berlabel (kelas + bounding box); analogi deteksi kucing/anjing. Deteksi objek = klasifikasi per objek + regresi koordinat kotak, jadi bukan murni klasifikasi |
| 4: cross-validation K-Fold tergrup | Pertemuan 6 - Evaluasi Model | Metode Cross-Validation (LOOCV, K-Fold, 10-Fold, Stratified) diadaptasi menjadi Stratified Group K-Fold: unit fold adalah video/kamera, hasil dilaporkan mean ± std |
| 5: evaluasi val/test | Pertemuan 6 - Evaluasi Model | Metrik klasifikasi (precision, recall, confusion matrix, PR curve) + metrik khusus detection (mAP@0.5, mAP@0.5:0.95) |
| 6-7: ekspor dan pemakaian `best.pt` | Pertemuan 3 & 6 - Deployment | Model masuk mesin produksi, catat input/output/eksepsi, pantau latency dan concept drift |

Pertemuan 5 (Unsupervised Learning: K-Means, asosiasi, Rand Index/Mutual Information) tidak dipakai: kelas `smoke`/`fire` dan lokasi objek sudah diketahui, jadi labelnya supervised. Metrik unsupervised di Pertemuan 6 (Rand Index, Mutual Information) juga tidak berlaku untuk deteksi.

Cross-validation (Pertemuan 6) tetap dijalankan, tapi tidak dalam bentuk K-fold acak per gambar: memutar fold di atas frame yang berasal dari rekaman sama membocorkan data. Notebook memakai **Stratified Group K-Fold** pada pool train+val (langkah 4) dengan video/kamera sebagai unit fold, lalu melaporkan mean ± std antar fold. Split `test` tetap menjadi holdout final yang tidak pernah ikut fold (langkah 5).

Urutan kerja sesuai materi: data preparation/cleaning (GIGO) → label supervised → train/validation/test → cross-validation K-Fold (mean ± std) → evaluasi *precision*, *recall*, `mAP@0.5`, `mAP@0.5:0.95` → deployment dan pantau concept drift.

## 0. Setup: dependensi, koneksi Drive, konfigurasi

Tahap ini belum menyentuh data maupun model - hanya menyiapkan lingkungan Colab dan parameter run.

In [ ]:
%pip -q install ultralytics pyyaml matplotlib

In [ ]:
# 0a. Mount Google Drive dan impor library.
from google.colab import drive
drive.mount('/content/drive')

from collections import Counter, defaultdict
from pathlib import Path
import hashlib, json, random, re, shutil, statistics

import cv2
import matplotlib.pyplot as plt
import torch
import yaml
from ultralytics import YOLO

In [ ]:
# 0b. Konfigurasi run: path kerja, arsitektur YOLO11, hyperparameter, device.
# Folder yang diminta; di Google Colab "My Drive" tersedia sebagai "MyDrive".
DATASET_SOURCE = Path('/content/drive/MyDrive/visikom')
WORK_DIR = Path('/content/fire-smoke-dataset')
RUNS_DIR = Path('/content/runs/fire-smoke')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/visikom/training-output/fire-smoke-yolo11n')

# Sesuaikan jika dataset belum berisi split. Jika gambar berasal dari video/CCTV,
# kelompokkan per video/kamera terlebih dahulu, lalu siapkan split YOLO manual.
ALLOW_IMAGE_LEVEL_SPLIT = False
CLASS_NAMES_FALLBACK = {0: 'smoke', 1: 'fire'}
# YOLO11 butuh ultralytics >= 8.3 (cell install di atas sudah mengambil versi terbaru).
BASE_MODEL = 'yolo11n.pt'
EPOCHS, IMAGE_SIZE, BATCH_SIZE = 50, 640, 16
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
SEED = 42

# Cross-validation (Pertemuan 6): K-Fold tergrup per sumber rekaman pada pool train+val.
# Biayanya CV_FOLDS x CV_EPOCHS epoch tambahan di luar training utama; CV_FOLDS=0 melewatinya.
CV_FOLDS, CV_EPOCHS = 3, 20

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (lebih lambat)')
print('Sumber:', DATASET_SOURCE)
assert DATASET_SOURCE.is_dir(), f'Folder tidak ditemukan: {DATASET_SOURCE}. Upload/ekstrak dataset di My Drive/visikom.'

## 1. Temukan dan siapkan split

Folder terlabel yang diterima: `train/images`, `train/labels`, `val/images`, `val/labels`, `test/images`, `test/labels` (nama `valid/` diterima). Alternatifnya, satu folder `images/` dan `labels/` dengan nama/struktur relatif sama; split acak 70/15/15 hanya berjalan bila `ALLOW_IMAGE_LEVEL_SPLIT=True`. Jika belum ada bounding box, anotasi dan ekspor ulang terlebih dahulu.

Script mengerjakan salinan lokal Colab, memeriksa pasangan label/gambar, membuang gambar rusak/duplikat persis dari salinan, dan menghentikan proses jika hash gambar yang sama ada di split berbeda. Data Drive asli tidak diubah. Split test harus disediakan dan independen. Bila split belum ada, image-level split opsional hanya layak untuk gambar independen; untuk video/CCTV, pisahkan berdasarkan sumber sebelum mengekspor dataset.

In [ ]:
# 1a. Temukan root dataset YOLO berlabel di Drive (train/val/test atau images+labels).
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def find_dataset_root(source):
    candidates = [source]
    for first in source.iterdir():
        if first.is_dir():
            candidates.append(first)
            candidates.extend(path for path in first.iterdir() if path.is_dir())
    for root in candidates:
        if (root / 'train/images').is_dir() and (root / 'train/labels').is_dir():
            val_name = next((name for name in ('val', 'valid', 'validation') if (root / f'{name}/images').is_dir() and (root / f'{name}/labels').is_dir()), None)
            if val_name:
                return root, val_name, True
    for root in candidates:
        if (root / 'images').is_dir() and (root / 'labels').is_dir():
            return root, None, False
    raise FileNotFoundError(
        f'Tidak ditemukan dataset YOLO berlabel di {source}. Ekspor anotasi YOLO Detection; '
        'butuh train/images+labels dan val/images+labels, atau images/+labels/ berpasangan.'
    )

In [ ]:
# 1b. Metode split acak 70/15/15 - hanya untuk gambar yang saling independen.
def split_files(files, fractions, seed=SEED):
    files = sorted(files)
    if len(files) < 3:
        raise ValueError('Minimal perlu 3 gambar terlabel untuk membuat train/val/test.')
    random.Random(seed).shuffle(files)
    n_test = max(1, round(len(files) * fractions[2]))
    n_val = max(1, round(len(files) * fractions[1]))
    if n_test + n_val >= len(files):
        n_test, n_val = 1, 1
    return {'test': files[:n_test], 'val': files[n_test:n_test+n_val], 'train': files[n_test+n_val:]}

In [ ]:
# 1c. Bangun split flat: salin pasangan gambar + label ke workspace Colab.
def make_flat_splits(root):
    if not ALLOW_IMAGE_LEVEL_SPLIT:
        raise ValueError('Split YOLO train/val/test belum tersedia. Siapkan split per video/kamera, atau ubah ALLOW_IMAGE_LEVEL_SPLIT=True untuk gambar yang saling independen.')
    images_root, labels_root = root / 'images', root / 'labels'
    images = sorted(p for p in images_root.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
    if not images:
        raise ValueError(f'Tidak ada gambar di {images_root}')
    missing = [p for p in images if not (labels_root / p.relative_to(images_root).with_suffix('.txt')).is_file()]
    if missing:
        raise ValueError(f'{len(missing)} gambar belum punya file label .txt; contoh: {missing[0]}. Buat label YOLO atau .txt kosong untuk background.')
    splits = split_files(images, (0.70, 0.15, 0.15))
    for split, paths in splits.items():
        for folder in ('images', 'labels'):
            (WORK_DIR / split / folder).mkdir(parents=True, exist_ok=True)
        for image in paths:
            relative = image.relative_to(images_root)
            target = WORK_DIR / split / 'images' / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(image, target)
            label = labels_root / relative.with_suffix('.txt')
            target_label = WORK_DIR / split / 'labels' / relative.with_suffix('.txt')
            target_label.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(label, target_label)
    print('Split per gambar (70/15/15); verifikasi independensi sumber sebelum menerima metrik.')

In [ ]:
# 1d. Orkestrasi split: pakai split bawaan dataset, atau buat split bila diizinkan.
DATASET_ROOT, SOURCE_VAL_NAME, HAS_PREDEFINED_SPLITS = find_dataset_root(DATASET_SOURCE)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

if HAS_PREDEFINED_SPLITS:
    shutil.copytree(DATASET_ROOT / 'train', WORK_DIR / 'train')
    shutil.copytree(DATASET_ROOT / SOURCE_VAL_NAME, WORK_DIR / SOURCE_VAL_NAME)
    if (DATASET_ROOT / 'test/images').is_dir() and (DATASET_ROOT / 'test/labels').is_dir():
        shutil.copytree(DATASET_ROOT / 'test', WORK_DIR / 'test')
    if SOURCE_VAL_NAME != 'val':
        (WORK_DIR / SOURCE_VAL_NAME).rename(WORK_DIR / 'val')
    print('Split berlabel ditemukan:', DATASET_ROOT)
    if not (WORK_DIR / 'test/images').is_dir() or not (WORK_DIR / 'test/labels').is_dir():
        raise FileNotFoundError('Split test independen wajib: tambahkan test/images dan test/labels. Jangan memakai ulang validation sebagai test setelah training/tuning.')
else:
    make_flat_splits(DATASET_ROOT)

In [ ]:
# 1e. Mapping ID kelas dari metadata ekspor dataset (fallback: 0=smoke, 1=fire).
# Gunakan mapping kelas dari metadata ekspor jika tersedia; selain itu 0=smoke, 1=fire.
source_yaml = next((p for p in (DATASET_ROOT / 'data.yaml', DATASET_ROOT / 'data.yml', DATASET_ROOT / 'dataset.yaml') if p.is_file()), None)
source_names = yaml.safe_load(source_yaml.read_text(encoding='utf-8')).get('names') if source_yaml else None
if isinstance(source_names, list):
    CLASS_NAMES = dict(enumerate(map(str, source_names)))
elif isinstance(source_names, dict):
    CLASS_NAMES = {int(class_id): str(name) for class_id, name in source_names.items()}
else:
    CLASS_NAMES = CLASS_NAMES_FALLBACK
if not CLASS_NAMES or sorted(CLASS_NAMES) != list(range(len(CLASS_NAMES))):
    raise ValueError(f'ID kelas harus berurutan mulai 0: {CLASS_NAMES}')
print('Mapping ID kelas:', CLASS_NAMES)
print('Dataset kerja lokal:', WORK_DIR)

## 2. Bersihkan, validasi anotasi, cek distribusi

Gambar rusak dan salinan duplikat dibuang dari workspace sementara; data asli di Drive tidak diubah. Anotasi hilang, format kotak salah, ID kelas asing, atau duplikat lintas split menghentikan notebook agar data leakage/noisy tidak tersamarkan.

In [ ]:
# 2a. Method validasi + cleaning untuk satu split (dipanggil lagi di 2b).
def validate_and_clean_split(split):
    images_dir = WORK_DIR / split / 'images'
    labels_dir = WORK_DIR / split / 'labels'
    if not images_dir.is_dir() or not labels_dir.is_dir():
        raise FileNotFoundError(f'{split} wajib berisi images/ dan labels/: {WORK_DIR / split}')
    images = sorted(p for p in images_dir.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
    if not images:
        raise ValueError(f'Tidak ada gambar valid pada split {split}')
    objects, negatives, bad, kept_hashes = Counter(), 0, [], set()
    for image in images:
        relative = image.relative_to(images_dir)
        label = labels_dir / relative.with_suffix('.txt')
        if not label.is_file():
            raise FileNotFoundError(f'Label hilang: {label}. Anotasikan objek; untuk background gunakan file .txt kosong.')
        pixels = cv2.imread(str(image))
        if pixels is None or pixels.size == 0:
            bad.append((image, label))
            continue
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        if digest in kept_hashes:
            bad.append((image, label))
            continue
        kept_hashes.add(digest)
        lines = [line for line in label.read_text(encoding='utf-8').splitlines() if line.strip()]
        negatives += not lines
        for line_number, line in enumerate(lines, 1):
            values = line.split()
            if len(values) != 5:
                raise ValueError(f'Format YOLO harus 5 kolom: {label}:{line_number}')
            try:
                class_id = int(values[0])
                box = [float(value) for value in values[1:]]
            except ValueError as error:
                raise ValueError(f'Label non-numerik: {label}:{line_number}') from error
            if class_id not in CLASS_NAMES:
                raise ValueError(f'ID kelas {class_id} tidak ada di mapping: {label}:{line_number}')
            if not all(0 <= value <= 1 for value in box) or box[2] <= 0 or box[3] <= 0:
                raise ValueError(f'Koordinat bbox harus ternormalisasi, lebar/tinggi positif: {label}:{line_number}')
            xc, yc, width, height = box
            if xc-width/2 < 0 or yc-height/2 < 0 or xc+width/2 > 1 or yc+height/2 > 1:
                raise ValueError(f'BBox keluar batas gambar: {label}:{line_number}')
            objects[class_id] += 1
    for image, label in bad:
        image.unlink(missing_ok=True)
        label.unlink(missing_ok=True)
    remaining = len(images) - len(bad)
    if remaining == 0:
        raise ValueError(f'Semua gambar pada split {split} rusak atau duplikat.')
    print(f'{split}: {remaining} gambar | background: {negatives} | objek: ' + ', '.join(f'{CLASS_NAMES[c]}={objects[c]}' for c in sorted(CLASS_NAMES)) + f' | dibersihkan: {len(bad)}')
    return {'images': remaining, 'objects': dict(objects), 'background': negatives}

In [ ]:
# 2b. Jalankan validasi untuk train/val/test dan cek distribusi kelas per split.
SPLIT_REPORT = {split: validate_and_clean_split(split) for split in ('train', 'val', 'test')}
for split, summary in SPLIT_REPORT.items():
    missing_classes = [CLASS_NAMES[class_id] for class_id in CLASS_NAMES if summary['objects'].get(class_id, 0) == 0]
    if missing_classes:
        raise ValueError(f'Split {split} tidak memiliki anotasi kelas {missing_classes}; seimbangkan ulang dataset sebelum training/evaluasi.')

In [ ]:
# 2c. Cek duplikat persis lintas split: gambar yang sama di train dan test = evaluasi bocor.
# Cek duplikat persis lintas split setelah cleaning; duplikat berarti evaluasi bocor.
hash_owner = {}
for split in ('train', 'val', 'test'):
    for image in (WORK_DIR / split / 'images').rglob('*'):
        if image.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        if digest in hash_owner:
            raise ValueError(f'Duplikat lintas split: {hash_owner[digest]} dan {image}. Bagi data berdasarkan rekaman/kamera yang independen.')
        hash_owner[digest] = image

In [ ]:
# 2d. Tulis konfigurasi dataset YOLO (data.yaml) yang dipakai training dan evaluasi.
DATA_YAML = Path('/content/fire-smoke-data.yaml')
DATA_CONFIG = {'path': str(WORK_DIR), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'names': CLASS_NAMES}
DATA_YAML.write_text(yaml.safe_dump(DATA_CONFIG, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('Konfigurasi YOLO:\n', DATA_YAML.read_text(encoding='utf-8'))
assert all(SPLIT_REPORT[split]['images'] > 0 for split in ('train', 'val', 'test'))

### 2e. Visualisasi ground-truth

Cek visual beberapa gambar train/val beserta kotak aslinya: cara tercepat menemukan anotasi bergeser atau ID kelas tertukar sebelum training.

In [ ]:
# Visualisasi sampel asli + ground-truth bbox; cek ID, posisi, dan ukuran anotasi sebelum training.
for split in ('train', 'val'):
    samples = [p for p in (WORK_DIR / split / 'images').rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES][:4]
    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5), squeeze=False)
    for ax, image_path in zip(axes[0], samples):
        image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]
        label_path = (WORK_DIR / split / 'labels' / image_path.relative_to(WORK_DIR / split / 'images')).with_suffix('.txt')
        for line in label_path.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            class_id, xc, yc, bw, bh = map(float, line.split())
            x, y = (xc-bw/2)*width, (yc-bh/2)*height
            rect = plt.Rectangle((x, y), bw*width, bh*height, fill=False, color='lime', linewidth=2)
            ax.add_patch(rect)
            ax.text(x, max(12, y-5), CLASS_NAMES[int(class_id)], color='white', backgroundcolor='green')
        ax.imshow(image); ax.set_title(f'{split}: {image_path.name}'); ax.axis('off')
    plt.tight_layout(); plt.show()

## 3. Latih YOLO11n (supervised transfer learning)

Checkpoint awal `yolo11n.pt` membawa fitur visual pralatih; model kemudian di-fine-tune menggunakan gambar dan bounding box berlabel. Tidak perlu resize atau normalisasi gambar manual: Ultralytics mengatur letterbox ke `IMAGE_SIZE=640`, skala piksel, dan transformasi koordinat anotasi; augmentasi seperti mosaic/flip hanya untuk train. `val` memantau generalisasi dan early stopping; `test` tidak digunakan untuk memilih epoch atau threshold.

In [ ]:
# 3a. Transfer learning: fine-tune `yolo11n.pt` pada dataset berlabel ini.
model = YOLO(BASE_MODEL)
train_metrics = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=DEVICE, workers=2, project=str(RUNS_DIR), name='fire-smoke-yolo11n',
    patience=15, seed=SEED, plots=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PT.is_file(), f'Checkpoint terbaik tidak ditemukan: {BEST_PT}'
print('Training artifacts:', RUN_DIR)
print('Best checkpoint:', BEST_PT)
print('Grafik training:', RUN_DIR / 'results.png')

## 4. Cross-validation: Stratified Group K-Fold (Pertemuan 6)

Satu angka holdout tidak menjawab pertanyaan Pertemuan 6 yang sebenarnya: **seberapa stabil estimasi performa ini?** K-Fold melatih model K kali, masing-masing dengan bagian berbeda sebagai holdout, lalu hasilnya dirata-ratakan. Notebook memakai bentuk yang aman untuk object detection:

- **Group** - unit pembagian adalah sumber rekaman (video/kamera), bukan gambar. Semua frame dari rekaman yang sama pindah fold bersamaan, sehingga tidak ada fold yang dilatih pada frame tetangga dari video yang sedang diujinya.
- **Stratified** - grup dikelompokkan lebih dulu berdasarkan komposisi kelas (`smoke`/`fire`/`background`), lalu didistribusikan ke fold yang paling sedikit menampung gambar, jadi tiap fold melihat proporsi kelas yang mirip.
- **K-Fold sejati** - untuk fold ke-*k*, `yolo11n.pt` dilatih ulang dari checkpoint pralatih pada K-1 bagian dan diuji pada bagian ke-*k*. Outputnya K angka + **mean ± std**.
- Pool fold adalah `train` + `val`; split `test` tidak pernah masuk daftar fold dan baru dipakai di langkah 5.

Biayanya nyata: `CV_FOLDS × CV_EPOCHS` epoch tambahan di luar training utama (default 3 × 20). Turunkan `CV_EPOCHS`/`CV_FOLDS` bila kuota Colab terbatas, atau set `CV_FOLDS=0` untuk melewati tahap ini. LOOCV dan 10-fold murni tidak dipakai apa adanya: holdout satu frame praktis identik dengan frame tetangganya di sisi training, sehingga variansnya palsu kecil.

Cara membaca: std kecil berarti performa konsisten lintas rekaman. std besar berarti satu angka holdout menyesatkan - tambah footage independen. Train tinggi tapi mean fold rendah = overfitting ke rekaman tertentu.

In [ ]:
# 4a. Helper fold: kelompokkan gambar per sumber rekaman, lalu bagi grup ke K fold secara terstratifikasi.
def source_group(image, split):
    """Unit fold = satu sumber rekaman (video/kamera), bukan satu gambar."""
    relative = image.relative_to(WORK_DIR / split / 'images')
    if len(relative.parts) > 1:
        return '/'.join(relative.parts[:-1])
    tokens = [token for token in re.split(r'[-_.\s]+', image.stem) if token]
    prefix = next((token for token in tokens
                   if not token.isdigit() and token.lower() not in ('frame', 'img', 'image', 'video', 'shot', 'clip')), None)
    return prefix.lower() if prefix else image.stem

def group_classes(members):
    classes = set()
    for split, image in members:
        label = (WORK_DIR / split / 'labels'
                / image.relative_to(WORK_DIR / split / 'images').with_suffix('.txt'))
        for line in label.read_text(encoding='utf-8').splitlines():
            if line.strip():
                classes.add(CLASS_NAMES[int(line.split()[0])])
    return tuple(sorted(classes)) if classes else ('background',)

def build_folds(groups, folds):
    """Stratified GroupKFold: satu grup tidak pernah terbelah dua sisi, komposisi kelas diratakan."""
    buckets = defaultdict(list)
    for key, members in groups.items():
        buckets[group_classes(members)].append(key)
    fold_of_group = {}
    fold_sizes = {index: 0 for index in range(folds)}
    for signature in sorted(buckets):
        for key in sorted(buckets[signature], key=lambda item: (-len(groups[item]), item)):
            target = min(fold_sizes, key=lambda index: (fold_sizes[index], index))
            fold_of_group[key] = target
            fold_sizes[target] += len(groups[key])
    return fold_of_group

In [ ]:
# 4b. Susun daftar fold dari pool train+val (split test tidak pernah ikut menjadi fold).
CV_POOL = defaultdict(list)
for split in ('train', 'val'):
    for image_path in sorted((WORK_DIR / split / 'images').rglob('*')):
        if image_path.suffix.lower() in IMAGE_SUFFIXES:
            CV_POOL[source_group(image_path, split)].append((split, image_path))
CV_DIR = RUN_DIR / 'cross-validation'
CV_FOLD_SPECS = []
if CV_FOLDS == 0:
    print('CV_FOLDS=0: cross-validation dilewati; hanya holdout train/val/test yang tersisa.')
elif CV_FOLDS < 2:
    raise ValueError('CV_FOLDS minimal 2 agar ada sisi latih dan sisi holdout (gunakan 0 untuk melewati).')
else:
    if len(CV_POOL) < CV_FOLDS:
        raise ValueError(f'Butuh minimal {CV_FOLDS} sumber rekaman independen untuk K-Fold; hanya ada {len(CV_POOL)}. Pisahkan dataset per video/kamera atau turunkan CV_FOLDS.')
    fold_of_group = build_folds(CV_POOL, CV_FOLDS)
    CV_DIR.mkdir(parents=True, exist_ok=True)
    for holdout in range(CV_FOLDS):
        holdout_groups = sorted(key for key, index in fold_of_group.items() if index == holdout)
        fit_groups = sorted(key for key, index in fold_of_group.items() if index != holdout)
        holdout_paths = [image for key in holdout_groups for _, image in CV_POOL[key]]
        fit_paths = [image for key in fit_groups for _, image in CV_POOL[key]]
        if set(fit_paths) & set(holdout_paths):
            raise ValueError(f'Fold {holdout} bocor: satu rekaman ada di sisi latih sekaligus holdout.')
        if any(path.relative_to(WORK_DIR).parts[0] == 'test' for path in fit_paths + holdout_paths):
            raise ValueError(f'Fold {holdout} menyentuh split test; cross-validation hanya boleh memakai train+val.')
        holdout_classes = set()
        for key in holdout_groups:
            holdout_classes.update(name for name in group_classes(CV_POOL[key]) if name != 'background')
        missing = sorted(set(CLASS_NAMES.values()) - holdout_classes)
        if missing:
            raise ValueError(f'Fold {holdout} tidak punya ground-truth kelas {missing}; seimbangkan grup rekaman sebelum cross-validation.')
        listing = CV_DIR / f'fold-{holdout}'
        listing.mkdir(parents=True, exist_ok=True)
        train_list, val_list = listing / 'train.txt', listing / 'val.txt'
        train_list.write_text('\n'.join(str(path) for path in fit_paths), encoding='utf-8')
        val_list.write_text('\n'.join(str(path) for path in holdout_paths), encoding='utf-8')
        fold_yaml = listing / 'data.yaml'
        fold_yaml.write_text(yaml.safe_dump({'path': str(WORK_DIR), 'train': str(train_list), 'val': str(val_list), 'names': CLASS_NAMES}, sort_keys=False, allow_unicode=True), encoding='utf-8')
        CV_FOLD_SPECS.append({'fold': holdout, 'yaml': fold_yaml, 'groups': holdout_groups, 'fit_images': len(fit_paths), 'holdout_images': len(holdout_paths)})
    total_images = sum(len(members) for members in CV_POOL.values())
    if len(CV_POOL) == total_images:
        print('PERINGATAN: tiap gambar dianggap sumber terpisah (tidak ada token video/kamera pada nama file). Fold image-level rawan kebocoran antar frame.')
    print(f'Cross-validation {CV_FOLDS}-Fold dari {len(CV_POOL)} sumber rekaman ({total_images} gambar pool train+val):')
    for spec in CV_FOLD_SPECS:
        print(f"  fold {spec['fold']}: latih {spec['fit_images']} gambar | holdout {spec['holdout_images']} gambar | grup: {', '.join(spec['groups'])}")

In [ ]:
# 4c. Latih ulang `yolo11n.pt` untuk tiap fold: holdout sebuah fold tidak pernah dilihat saat training fold itu.
CV_ROWS = []
for spec in CV_FOLD_SPECS:
    fold_model = YOLO(BASE_MODEL)
    fold_model.train(
        data=str(spec['yaml']), epochs=CV_EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=DEVICE, workers=2, project=str(CV_DIR), name=f"fold-{spec['fold']}",
        patience=10, seed=SEED + spec['fold'],
    )
    fold_best_pt = Path(fold_model.trainer.save_dir) / 'weights' / 'best.pt'
    if not fold_best_pt.is_file():
        raise FileNotFoundError(f"Checkpoint fold {spec['fold']} tidak ditemukan: {fold_best_pt}")
    fold_eval = YOLO(str(fold_best_pt)).val(
        data=str(spec['yaml']), split='val', imgsz=IMAGE_SIZE, device=DEVICE, plots=True,
        project=str(CV_DIR), name=f"fold-{spec['fold']}-holdout", exist_ok=True,
    )
    row = {'fold': spec['fold'], 'holdout_groups': spec['groups'], 'holdout_images': spec['holdout_images']}
    for metric, key in (('map50', 'metrics/mAP50(B)'), ('map50_95', 'metrics/mAP50-95(B)'), ('precision', 'metrics/precision(B)'), ('recall', 'metrics/recall(B)')):
        row[metric] = float(fold_eval.results_dict[key])
    present_ids = list(map(int, fold_eval.box.ap_class_index))
    row['per_class'] = {}
    for class_id, class_name in CLASS_NAMES.items():
        if class_id in present_ids:
            precision, recall, ap50, ap = fold_eval.box.class_result(present_ids.index(class_id))
            row['per_class'][class_name] = {'precision': float(precision), 'recall': float(recall), 'map50': float(ap50), 'map50_95': float(ap)}
    CV_ROWS.append(row)
    print(f"fold {spec['fold']}: mAP@0.5={row['map50']:.4f} | mAP@0.5:0.95={row['map50_95']:.4f} | P={row['precision']:.4f} | R={row['recall']:.4f}")

In [ ]:
# 4d. Agregat antar fold menjadi mean ± std: ukuran keyakinan, bukan sekadar satu angka.
def mean_std(values):
    return statistics.fmean(values), (statistics.stdev(values) if len(values) > 1 else 0.0)

CV_METRICS = ('map50', 'map50_95', 'precision', 'recall')
if not CV_ROWS:
    print('Tidak ada hasil fold untuk diagregasi.')
else:
    CV_SUMMARY = {'cv_folds': CV_FOLDS, 'epochs_per_fold': CV_EPOCHS, 'base_model': BASE_MODEL,
                  'pool': 'train+val (split test tidak ikut fold)', 'recordings': len(CV_POOL),
                  'mean_std': {}, 'per_class': {}, 'folds': CV_ROWS}
    print(f'Hasil cross-validation {CV_FOLDS}-Fold tergrup per sumber rekaman:')
    for metric in CV_METRICS:
        mean, std = mean_std([row[metric] for row in CV_ROWS])
        CV_SUMMARY['mean_std'][metric] = {'mean': mean, 'std': std}
        print(f'  {metric}: {mean:.4f} ± {std:.4f}')
    per_class_values = defaultdict(list)
    for row in CV_ROWS:
        for class_name, values in row['per_class'].items():
            per_class_values[class_name].append(values)
    for class_name, items in sorted(per_class_values.items()):
        CV_SUMMARY['per_class'][class_name] = {metric: dict(zip(('mean', 'std'), mean_std([item[metric] for item in items]))) for metric in CV_METRICS}
        summary_class = CV_SUMMARY['per_class'][class_name]
        print(f"  {class_name}: mAP@0.5={summary_class['map50']['mean']:.4f} ± {summary_class['map50']['std']:.4f} | recall={summary_class['recall']['mean']:.4f} ± {summary_class['recall']['std']:.4f}")
    map50_spread = CV_SUMMARY['mean_std']['map50']['std']
    if map50_spread <= 0.05:
        print(f'Selisih antar fold kecil (±{map50_spread:.4f}): estimasi performa stabil lintas rekaman.')
    else:
        print(f'Selisih antar fold besar (±{map50_spread:.4f}): satu angka holdout menyesatkan; tambah rekaman independen atau perbaiki ketidakseimbangan antar kamera.')
    (CV_DIR / 'cv-metrics.json').write_text(json.dumps(CV_SUMMARY, indent=2), encoding='utf-8')
    print('Laporan cross-validation:', CV_DIR / 'cv-metrics.json')

In [ ]:
# 4e. Grafik hasil per fold: stabilitas estimasi lebih mudah dibaca secara visual.
if CV_ROWS:
    fig, ax = plt.subplots(figsize=(7, 4))
    labels = [f"fold {row['fold']}" for row in CV_ROWS]
    for metric, marker in (('map50', 'o'), ('map50_95', 's')):
        values = [row[metric] for row in CV_ROWS]
        mean, std = mean_std(values)
        ax.errorbar(range(len(values)), values, yerr=std, fmt=marker, capsize=4, label=f'{metric} per fold')
        ax.axhline(mean, linestyle='--', alpha=0.45, label=f'{metric} mean {mean:.3f}')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel('mAP')
    ax.legend(fontsize=8)
    ax.set_title(f'Cross-validation {CV_FOLDS}-Fold (tergrup per sumber rekaman)')
    fig.tight_layout()
    fig.savefig(CV_DIR / 'cv-folds.png', dpi=120)
    plt.show()

## 5. Evaluasi final pada test split

Precision tinggi berarti lebih sedikit alarm palsu; recall tinggi berarti lebih sedikit api/asap terlewat. `mAP@0.5` mengukur deteksi pada IoU 0.5; `mAP@0.5:0.95` lebih ketat, dirata-ratakan atas IoU 0.50–0.95. Tinjau metrik **per kelas**, confusion matrix, PR curve, dan contoh prediksi—angka global dapat menyembunyikan kegagalan kelas asap/api yang langka. Jalankan test setelah model/threshold dipilih menggunakan validation; jangan gunakan test berulang untuk tuning. Angka di bawah adalah holdout tunggal - laporkan berdampingan dengan mean ± std cross-validation dari langkah 4.

In [ ]:
# 5a. Evaluasi final pada split test (satu kali, setelah model/threshold dipilih lewat validation).
best_model = YOLO(str(BEST_PT))
test_results = best_model.val(
    data=str(DATA_YAML), split='test', imgsz=IMAGE_SIZE, device=DEVICE,
    plots=True, project=str(RUN_DIR), name='test-evaluation', exist_ok=True,
)

In [ ]:
# 5b. Metrik per kelas + laporan `metrics.json`.
present_ids = list(map(int, test_results.box.ap_class_index))
per_class = []
for class_id, class_name in CLASS_NAMES.items():
    if class_id in present_ids:
        index = present_ids.index(class_id)
        precision, recall, ap50, ap = test_results.box.class_result(index)
    else:
        precision = recall = ap50 = ap = 0.0
    per_class.append({'class': class_name, 'precision': float(precision), 'recall': float(recall), 'map50': float(ap50), 'map50_95': float(ap), 'ground_truth_objects': int(SPLIT_REPORT['test']['objects'].get(class_id, 0))})

summary = {
    'weights': str(BEST_PT), 'test_images': SPLIT_REPORT['test']['images'],
    'device': str(DEVICE), 'metrics': {key: float(value) for key, value in test_results.results_dict.items()},
    'per_class': per_class,
}
TEST_DIR = RUN_DIR / 'test-evaluation'
(TEST_DIR / 'metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Confusion matrix / PR curves:', TEST_DIR)
assert len(per_class) == len(CLASS_NAMES) and all(item['ground_truth_objects'] > 0 for item in per_class), 'Setiap kelas perlu anotasi ground-truth pada test untuk evaluasi yang bermakna.'

### 5c. Inspeksi visual dan kecepatan inference

Lihat contoh prediksi dan ukur latency per frame; angka agregat saja tidak membuktikan model layak dipakai realtime di CCTV.

In [ ]:
# Inspect prediksi visual dan ukur kecepatan pada perangkat inference yang tersedia.
sample_images = [p for p in (WORK_DIR / 'test/images').rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES][:4]
for image_path in sample_images:
    prediction = best_model.predict(str(image_path), imgsz=IMAGE_SIZE, device=DEVICE, conf=0.25, verbose=False)[0]
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(prediction.plot(), cv2.COLOR_BGR2RGB))
    plt.title(f'{image_path.name} | latency: {prediction.speed["inference"]:.1f} ms/frame')
    plt.axis('off'); plt.show()
print('Latency ms/image (preprocess, inference, postprocess):', test_results.speed)

## 6. Ekspor artefak ke My Drive

Yang disalin hanya artefak model (checkpoint, metrik, kurva). Dataset sumber di Drive tetap utuh.

In [ ]:
# Ekspor checkpoint dan metrik ke My Drive. Sumber dataset tetap utuh; hanya artefak model disalin.
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_PT, DRIVE_OUTPUT / 'best.pt')
shutil.copy2(TEST_DIR / 'metrics.json', DRIVE_OUTPUT / 'test-metrics.json')
if CV_ROWS:
    for cv_artifact in (CV_DIR / 'cv-metrics.json', CV_DIR / 'cv-folds.png'):
        if cv_artifact.is_file(): shutil.copy2(cv_artifact, DRIVE_OUTPUT / cv_artifact.name)
for filename in ('results.csv', 'args.yaml', 'results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png'):
    artifact = RUN_DIR / filename
    if artifact.is_file(): shutil.copy2(artifact, DRIVE_OUTPUT / filename)
    artifact = TEST_DIR / filename
    if artifact.is_file(): shutil.copy2(artifact, DRIVE_OUTPUT / f'test-{filename}')
print('Model dan laporan:', DRIVE_OUTPUT)
print("Download model: from google.colab import files; files.download(str(DRIVE_OUTPUT / 'best.pt'))")

## 7. Pakai `best.pt` di aplikasi

1. Unduh `best.pt` dari `My Drive/visikom/training-output/fire-smoke-yolo11n/`.
2. Salin ke `fire-smoke-detection/models/weights/best.pt`, atau ke `models/fire-smoke-yolo11n/best.pt` sebagai checkpoint resmi. File bobot besar diabaikan Git.
3. Dari folder `backend/`: `pip install -r requirements-yolo.txt`, lalu `uvicorn app.main:app --reload --host 0.0.0.0 --port 8000`.
4. Cek `GET http://localhost:8000/api/v1/health`; pastikan engine YOLO aktif. Atur `NEXT_PUBLIC_BACKEND_URL=http://localhost:8000` di `frontend/.env.local`. Gambar memakai `POST /api/v1/detect/image`; hasil berisi kelas, confidence, dan bbox ternormalisasi.
5. Untuk CCTV, daftarkan alamat stream melalui `POST /api/v1/cameras` lalu sambungkan frontend ke `WS /api/v1/ws/cctv/{camera_id}`. Backend harus berjalan di mesin/jaringan yang dapat mengakses RTSP. Bounding-box live digambar frontend dari deteksi per-frame.

### Baca hasil dan lanjutkan
- **Varian antar fold:** gunakan mean ± std cross-validation (langkah 4) sebagai angka yang dilaporkan; holdout test tunggal hanya valid bila selisih antar fold kecil.
- **Underfitting:** train dan validation sama-sama rendah → tambah data berlabel/epoch atau coba model lebih besar.
- **Overfitting:** train tinggi, validation/test jauh lebih rendah → cek data leakage, variasi data dan anotasi; tambah data independen/augmentasi.
- **Recall api/asap rendah:** tinjau FN per kelas, contoh malam/jarak jauh, anotasi, imbalance; pilih threshold dari validation sesuai biaya false negative.
- **Realtime:** ukur latency end-to-end di perangkat CCTV. Jika lambat, coba YOLO ukuran nano, resolusi input lebih kecil, atau GPU. Monitor input, prediksi, latency, dan error setelah deployment; perubahan distribusi kamera/cuaca/pencahayaan adalah concept drift.